In [1]:
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import (
    IntegerType, LongType, DecimalType, TimestampType, BooleanType, StringType
)
from datetime import datetime
 
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

TIMESTAMP_FORMAT = "yyyy-MM-dd HH:mm:ss"

TIMESTAMP_FORMAT_FALLBACKS = [
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd",
    "M/d/yyyy H:mm",
    "M/d/yyyy H:mm:ss",
]

StatementMeta(, 13f7504e-0661-4cbd-b5cc-d0635741484f, 3, Finished, Available, Finished, False)

In [2]:
TABLE_CONFIG = {
    "customers": {
        "primary_key": ["customer_id"],
        "silver_table": "customers_T",
        "string_columns": ["first_name", "last_name", "phone", "city", "province", "country"],
        "email_columns": ["email"],
        "timestamp_columns": ["created_timestamp", "updated_timestamp"],
        "integer_columns": ["customer_id"],
        "decimal_columns": {},
        "flag_columns": ["is_active"],
    },
    "employees": {
        "primary_key": ["employee_id"],
        "silver_table": "employees_T",
        "string_columns": ["first_name", "last_name", "job_title"],
        "email_columns": ["email"],
        "timestamp_columns": ["created_timestamp", "updated_timestamp"],
        "integer_columns": ["employee_id", "store_id"],
        "decimal_columns": {"salary": (18, 2)},
        "flag_columns": ["is_active"],
    },
    "order_items": {
        "primary_key": ["order_item_id"],
        "silver_table": "order_items_T",
        "string_columns": [],
        "email_columns": [],
        "timestamp_columns": ["created_timestamp", "updated_timestamp"],
        "integer_columns": ["order_item_id", "order_id", "product_id", "quantity"],
        "decimal_columns": {"unit_price": (18, 2), "line_amount": (18, 2)},
        "flag_columns": ["is_active"],
    },
    "orders": {
        "primary_key": ["order_id"],
        "silver_table": "orders_T",
        "string_columns": ["payment_method", "order_status"],
        "email_columns": [],
        "timestamp_columns": ["order_timestamp", "created_timestamp", "updated_timestamp"],
        "integer_columns": ["order_id", "customer_id", "store_id"],
        "decimal_columns": {"total_amount": (18, 2)},
        "flag_columns": ["is_active"],
    },
    "products": {
        "primary_key": ["product_id"],
        "silver_table": "products_T",
        "string_columns": ["product_name", "category", "brand"],
        "email_columns": [],
        "timestamp_columns": ["created_timestamp", "updated_timestamp"],
        "integer_columns": ["product_id"],
        "decimal_columns": {"price": (18, 2)},
        "flag_columns": ["is_active"],
    },
    "stores": {
        "primary_key": ["store_id"],
        "silver_table": "stores_T",
        "string_columns": ["store_name", "city", "province", "country"],
        "email_columns": [],
        "timestamp_columns": ["created_timestamp", "updated_timestamp"],
        "integer_columns": ["store_id"],
        "decimal_columns": {},
        "flag_columns": ["is_active"],
    },
}

StatementMeta(, 13f7504e-0661-4cbd-b5cc-d0635741484f, 4, Finished, Available, Finished, False)

In [9]:
%%sql
select * from silver.orders_t limit 3;

StatementMeta(, 13f7504e-0661-4cbd-b5cc-d0635741484f, 11, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 11 fields>

In [4]:
def read_bronze_table(table_name: str, schema: str = BRONZE_SCHEMA) -> DataFrame:
    full_name = f"{schema}.{table_name}"
    print(f"[READ] {full_name}")
    return spark.read.table(full_name)

def existing_cols(df: DataFrame, cols) -> list:
    return [c for c in cols if c in df.columns]

def remove_null_primary_keys(df: DataFrame, primary_key: list) -> DataFrame:
    pk_cols = existing_cols(df, primary_key)
    if not pk_cols:
        return df

    condition = None
    for c in pk_cols:
        cond = F.col(c).isNotNull() & (F.trim(F.col(c).cast(StringType())) != "")
        condition = cond if condition is None else (condition & cond)

    return df.filter(condition)

def remove_duplicates(df: DataFrame, primary_key: list) -> DataFrame:
    pk_cols = existing_cols(df, primary_key)
    if not pk_cols:
        return df.dropDuplicates()
    return df.dropDuplicates(pk_cols)

def trim_string_columns(df: DataFrame, string_columns: list = None) -> DataFrame:
    cols = string_columns if string_columns else [
        f.name for f in df.schema.fields if isinstance(f.dataType, StringType)
    ]
    cols = existing_cols(df, cols)

    for c in cols:
        df = df.withColumn(c, F.trim(F.col(c)))

    return df

def standardize_string_formatting(df: DataFrame, string_columns: list) -> DataFrame:
    cols = existing_cols(df, string_columns)

    for c in cols:
        df = df.withColumn(
            c,
            F.when(F.trim(F.col(c)) == "", None)
             .otherwise(F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))
        )

    return df

def lowercase_email_columns(df: DataFrame, email_columns: list) -> DataFrame:
    cols = existing_cols(df, email_columns)

    for c in cols:
        df = df.withColumn(c, F.lower(F.trim(F.col(c))))

    return df

def cast_timestamp_columns(
    df: DataFrame,
    timestamp_columns: list,
    fmt: str = TIMESTAMP_FORMAT,
    fallback_formats: list = None
) -> DataFrame:
    fallback_formats = (
        fallback_formats
        if fallback_formats is not None
        else TIMESTAMP_FORMAT_FALLBACKS
    )

    cols = existing_cols(df, timestamp_columns)

    for c in cols:
        raw = F.trim(F.col(c).cast(StringType()))
        parsed = F.to_timestamp(raw, fmt)

        for fallback_fmt in fallback_formats:
            parsed = F.coalesce(parsed, F.to_timestamp(raw, fallback_fmt))

        df = df.withColumn(c, parsed)

    return df

def cast_numeric_columns(
    df: DataFrame,
    integer_columns: list,
    decimal_columns: dict
) -> DataFrame:
    int_cols = existing_cols(df, integer_columns)

    for c in int_cols:
        df = df.withColumn(
            c,
            F.trim(F.col(c).cast(StringType())).cast(IntegerType())
        )

    dec_cols = existing_cols(df, list(decimal_columns.keys()))

    for c in dec_cols:
        precision, scale = decimal_columns[c]
        df = df.withColumn(
            c,
            F.trim(F.col(c).cast(StringType())).cast(
                DecimalType(precision, scale)
            )
        )

    return df

def convert_flag_columns(df: DataFrame, flag_columns: list) -> DataFrame:
    cols = existing_cols(df, flag_columns)

    for c in cols:
        normalized = F.upper(F.trim(F.col(c).cast(StringType())))

        df = df.withColumn(
            c,
            F.when(normalized.isin("Y", "YES", "TRUE", "1"), F.lit(True))
             .when(normalized.isin("N", "NO", "FALSE", "0"), F.lit(False))
             .otherwise(None)
             .cast(BooleanType())
        )

    return df

def add_metadata_columns(df: DataFrame) -> DataFrame:
    return df.withColumn("processed_at", F.current_timestamp())

def data_quality_check(
    df_before: DataFrame,
    df_after: DataFrame,
    table_name: str
) -> dict:
    count_before = df_before.count()
    count_after = df_after.count()
    dropped = count_before - count_after

    summary = {
        "table": table_name,
        "rows_before": count_before,
        "rows_after": count_after,
        "rows_dropped": dropped,
        "checked_at": datetime.now().isoformat(),
    }

    print(
        f"[DQ] {table_name}: before={count_before} "
        f"after={count_after} dropped={dropped}"
    )

    return summary

def write_silver_table(
    df: DataFrame,
    table_name: str,
    schema: str = SILVER_SCHEMA
) -> None:
    full_name = f"{schema}.{table_name}"
    print(f"[WRITE] {full_name}")

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(full_name)
    )

StatementMeta(, 13f7504e-0661-4cbd-b5cc-d0635741484f, 6, Finished, Available, Finished, False)

In [5]:
def process_bronze_to_silver(bronze_table_name: str, config: dict) -> dict:
    
 
    cfg = config[bronze_table_name]
 
    df_raw = read_bronze_table(bronze_table_name)
    row_count_before = df_raw.count()
 
    df = df_raw
    df = remove_null_primary_keys(df, cfg["primary_key"])
    df = remove_duplicates(df, cfg["primary_key"])
    df = trim_string_columns(df)  
    df = standardize_string_formatting(df, cfg.get("string_columns", []))
    df = lowercase_email_columns(df, cfg.get("email_columns", []))
    df = cast_timestamp_columns(df, cfg.get("timestamp_columns", []))
    df = cast_numeric_columns(df, cfg.get("integer_columns", []), cfg.get("decimal_columns", {}))
    df = convert_flag_columns(df, cfg.get("flag_columns", []))
    df = add_metadata_columns(df)
 
    dq_summary = data_quality_check(df_raw, df, bronze_table_name)
    dq_summary["rows_before"] = row_count_before  
 
    write_silver_table(df, cfg["silver_table"])
 
    return dq_summary
 

StatementMeta(, 13f7504e-0661-4cbd-b5cc-d0635741484f, 7, Finished, Available, Finished, False)

In [6]:

dq_results = []
 
for bronze_table in TABLE_CONFIG.keys():
    try:
        result = process_bronze_to_silver(bronze_table, TABLE_CONFIG)
        dq_results.append(result)
    except Exception as e:
        print(f"[ERROR] Failed processing '{bronze_table}': {e}")
        dq_results.append({
            "table": bronze_table,
            "rows_before": None,
            "rows_after": None,
            "rows_dropped": None,
            "error": str(e),
            "checked_at": datetime.now().isoformat(),
        })
 

StatementMeta(, 13f7504e-0661-4cbd-b5cc-d0635741484f, 8, Finished, Available, Finished, False)

[READ] bronze.customers
[DQ] customers: before=2000 after=2000 dropped=0
[WRITE] silver.customers_T
[READ] bronze.employees
[DQ] employees: before=250 after=250 dropped=0
[WRITE] silver.employees_T
[READ] bronze.order_items
[DQ] order_items: before=30021 after=30021 dropped=0
[WRITE] silver.order_items_T
[READ] bronze.orders
[DQ] orders: before=10000 after=10000 dropped=0
[WRITE] silver.orders_T
[READ] bronze.products
[DQ] products: before=500 after=500 dropped=0
[WRITE] silver.products_T
[READ] bronze.stores
[DQ] stores: before=25 after=25 dropped=0
[WRITE] silver.stores_T


In [7]:
dq_df = spark.createDataFrame(dq_results)
display(dq_df)
 
(
    dq_df.write
      .format("delta")
      .mode("append")
      .saveAsTable(f"{SILVER_SCHEMA}.dq_log_technical_layer")
)
 
print("Silver Technical layer load complete.")

StatementMeta(, 13f7504e-0661-4cbd-b5cc-d0635741484f, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 157e7ad3-363f-41fe-9b3d-a340e9899916)

Silver Technical layer load complete.
